In [ ]:
import os
import torch
import sys
import h5py
from tqdm import tqdm
from embed import SkipLSTM, read_seq, Uniprot21


In [ ]:
spe = "yeast"

data_dir = "ppi-data"

# from google.colab import drive
#
# drive.mount('/content/drive')
# data_dir = "drive/MyDrive/ppi-data"

state_dict_path = os.path.join(data_dir, "embedding/dscript_lm_v1.pt")
seq_file = os.path.join(data_dir, spe, "seq/protein.dictionary.tsv")
embedding_h5 = os.path.join(data_dir, spe, "seq/lm_v1.embedding.h5")

use_cuda = torch.cuda.is_available()


In [ ]:
state_dict = torch.load(state_dict_path, weights_only=True)
model = SkipLSTM(21, 100, 1024, 3)
model.load_state_dict(state_dict)

torch.nn.init.normal_(model.proj.weight)
model.proj.bias = torch.nn.Parameter(torch.zeros(100))
if use_cuda:
    model = model.cuda()

model.eval()

In [ ]:
names, seqs = read_seq(seq_file)
alphabet = Uniprot21()
encoded_seqs = []
for s in seqs:
    es = torch.from_numpy(alphabet.encode(s.encode("utf-8")))
    if use_cuda:
        es = es.cuda()
    encoded_seqs.append(es)

In [ ]:

with torch.no_grad(), h5py.File(embedding_h5, "w") as h5fi:
    try:
        for name, x in tqdm(zip(names, encoded_seqs, strict=False), total=len(names)):
            if name not in h5fi:
                x = x.long().unsqueeze(0)
                z = model.transform(x)
                z = z.squeeze(0)
                dset = h5fi.require_dataset(
                    name,
                    shape=z.shape,
                    dtype="float32",
                    compression="lzf",
                )
                dset[:] = z.cpu().numpy()
    except KeyboardInterrupt:
        sys.exit(1)